# Gemini を使用した Function Calling（関数呼び出し）

**学習目標**

1. Function Calling と関連するユースケースについて学ぶ
1. Gemini Pro で Function Calling を実装する方法を学ぶ
1. チャットセッションで関数呼び出しを処理するためのパターンを学ぶ
1. さまざまな状況やユースケースで Function Calling をどのように活用できるかを学ぶ

Function Calling（関数呼び出し）を使用すると、デベロッパーはカスタム関数を定義して Gemini に提供できます。クエリの処理中、Gemini は特定のデータ処理タスクをこれらの関数に委任することを選択できます。Gemini 自体が直接これらの関数を呼び出すわけではなく、選択された関数の名前と、その関数を呼び出す際に渡すべき引数を含む構造化データを出力します。この出力を使用して、外部 API の呼び出し、計算の実行、構造化データの抽出などのタスクを実行できます。その後、関数のレスポンスをモデルに送り返すことで、モデルがクエリに対する最終的な回答を完成させることができます。

<img src="https://cloud.google.com/static/vertex-ai/generative-ai/docs/multimodal/images/function-calling.png" alt="Function Calling" class="center">

In [ ]:
from typing import Any, Callable, Optional, Tuple, Union

from google import genai
from google.cloud import bigquery
from google.genai.types import (
    FunctionDeclaration,
    GenerateContentConfig,
    GenerateContentResponse,
    Part,
    Schema,
    Tool,
    ThinkingConfig,
    ThinkingLevel,
)

%load_ext google.cloud.bigquery

In [ ]:
REGION = "us-central1"
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]

MODEL = "gemini-3.5-flash"

client = genai.Client(vertexai=True, location="global")

## Function Calling を使用したチャットセッション
まず、チャットセッション内で Function Calling をどのように実装できるかを考えてみましょう。基本的に、モデルが関数呼び出しを返したとき、ユーザーにそのまま返すのではなく、提供された引数を使用して指定された関数を実行する Python 関数を呼び出し、その結果をモデルに送り返す必要があります。これは複数回発生する可能性があります（例: モデルが関数呼び出しを返す -> 関数呼び出しのレスポンスがモデルにフィードバックされる -> モデルが別の関数呼び出しを返す -> ...）。

ここでの目標は、`send_message` メソッド内に推論ループ（Reasoning Loop）を実装した、サンプルチャットセッション用のシンプルなクラスを作成することです。このクラスは以下を指定してインスタンス化します。
1) `tools`: `Tool` のリスト
2) `tool_handler_fn`: 呼び出されたときに関数呼び出し名（str）と関数呼び出し引数（dict）を受け取る Python の呼び出し可能オブジェクト。この関数は、関数呼び出し自体のロジックを実装し、結果を返す必要があります。

たとえば、データベースからの読み取りと書き込みを行う関数呼び出しを持つツールがある場合、`tool_handler_fn` は次のようになります。

```python
def tool_handler_fn(fn_name, fn_args):
    """関数呼び出し read_row にはパラメータ row_id があり、write_row にはパラメータ row があると想定"""
    if fn_name == "read_row":
        result = db.read_row(fn_args["row_id"])
    elif fn_name == "write_row":
        result = db.write_row(fn_args["row"])
    return result 
```

In [ ]:
class ChatAgent:
    def __init__(
        self,
        tools: list[Tool],
        tool_handler_fn: Callable[[str, dict], Any],
        max_iterative_calls: int = 5,
    ):
        self.tools = tools
        self.tool_handler_fn = tool_handler_fn
        self.chat_session = client.chats.create(
            model=MODEL,
            config=GenerateContentConfig(
                tools=tools,
                thinking_config=ThinkingConfig(
                    # オプション: MINIMAL, LOW, MEDIUM, HIGH
                    thinking_level=ThinkingLevel.LOW 
                )
            ),
        )
        self.max_iterative_calls = 5

    def send_message(self, message: str) -> GenerateContentResponse:
        response = self.chat_session.send_message(message)
        # 関数呼び出しがトリガーされなかった場合は None
        fn_calls = response.function_calls

        num_calls = 0
        # 推論ループ。fn_calls が空の場合はここに入らず、
        # 単にレスポンスを返します
        while fn_calls:
            if num_calls > self.max_iterative_calls:
                break

            # 関数呼び出しを処理
            fn_call_responses = []
            for fn_call in fn_calls:
                response = self.tool_handler_fn(
                    fn_call.name, dict(fn_call.args)
                )
                fn_call_responses.append(
                    Part.from_function_response(
                        name=fn_call.name,
                        response={
                            "content": response,
                        },
                    ),
                )
                num_calls += 1

            # 関数呼び出しの結果をモデルに送り返す
            response = self.chat_session.send_message(fn_call_responses)

            # レスポンスが別の関数呼び出しである場合、
            # 推論ループにとどまり、関数の呼び出しを続けます
            fn_calls = response.function_calls

        return response

## シンプルな API の例
チャットセッションで Function Calling を使用する方法がわかったので、一般的なユースケースを実装してみましょう。ユーザーが特定の場所の現在の天気を尋ねたときに、API を呼び出して取得したいとします。これには、現在の天気が尋ねられていることを識別し、API リクエストに必要な場所を抽出するメカニズムが必要です。

Function Calling を使用すると、これは非常にシンプルになります。意図と必要なパラメータを含む関数宣言（Function Declaration）を定義するだけです。

In [ ]:
current_weather_func = FunctionDeclaration(
    name="current_weather",
    description="Get the current weather at a specified location",
    parameters=Schema(
        type="OBJECT",
        properties={
            "location": Schema(
                type="STRING",
                description="Location",
            ),
        },
        required=["location"],
    ),
)

# 気象 API を呼び出す関数をシミュレート


def current_weather(location: str) -> dict:
    print("Executing current_weather function...")
    api_response = {
        "location": "New York City",
        "temperature": "55 degrees (F)",
        "wind": "8 mph",
        "wind_direction": "West",
        "skies": "clear/sunny",
        "chance_of_rain": "0%",
    }
    return api_response

単一の関数宣言を含む `Tool` をインスタンス化し、モデルが関数呼び出しを返したときに呼び出されるツールハンドラー関数を作成します。その後、その `Tool` を使用してモデルをインスタンス化します。

In [ ]:
# ツールは 1 つまたは複数の関数をラップできます
weather_tool = Tool(
    function_declarations=[current_weather_func],
)

# 気象ツールを使用してチャットをインスタンス化
chat = client.chats.create(
    model=MODEL, config=GenerateContentConfig(tools=[weather_tool])
)

関数呼び出しのレスポンスがどのように見えるかを確認するため、`ChatAgent` を使わずにモデルにチャットメッセージを送信してみます。

In [ ]:
response = chat.send_message("What is the weather like in New York City?")
response

In [ ]:
response.function_calls

テキストレスポンスを返す代わりに、Gemini が呼び出すべき関数名と渡すべき引数を返していることに注目してください。次に、Gemini が関数呼び出しを返したときにいつでも関数名と引数を渡せるように、`ChatAgent` をインスタンス化する際に渡すハンドラー関数を実装します。

In [ ]:
def weather_tool_handler_fn(fn_name: str, fn_args: dict) -> dict:
    if fn_name == "current_weather":
        return current_weather(fn_args["location"])
    else:
        raise ValueError(f"Unknown function call: {fn_name}")


chat = ChatAgent(tools=[weather_tool], tool_handler_fn=weather_tool_handler_fn)
response = chat.send_message("What is the weather like in New York City?")
response.text

チャット履歴を確認すると、関数呼び出しが返され、ハンドラー関数が呼び出され、API をシミュレートする Python 関数が呼び出されたことがわかります。そのレスポンスがモデルに送り返され、天気に関する回答に組み込まれました。

In [ ]:
chat.chat_session.get_history()

## 数学演算を実行するための Function Calling
Function Calling は、LLM が長い間苦手としてきた分野である「数学・計算」にも役立ちます。言語モデルは自然言語の深く洞察に満ちた表現を構築しますが、（正確かつ一貫して）数学的な計算を実行する能力に欠けていることがよくあります。計算が必要なタイミングを特定し、実際にその演算を実行する関数を呼び出すツールを作成することで、一貫性と正確性を確保できます。

シンプルな計算演算（加算、減算、乗算、除算）の関数宣言を作成します。

In [ ]:
parameters = Schema(
    type="OBJECT",
    properties={
        "first_number": Schema(
            type="NUMBER",
            description="First number",
        ),
        "second_number": Schema(
            type="NUMBER",
            description="Second_number number",
        ),
    },
    required=["first_number", "second_number"],
)

add_two_numbers_func = FunctionDeclaration(
    name="add_two_numbers",
    description="Add two numbers together",
    parameters=parameters,
)


subtract_two_numbers_func = FunctionDeclaration(
    name="subtract_two_numbers",
    description="Subtract second_number from first_number",
    parameters=parameters,
)


multiply_two_numbers_func = FunctionDeclaration(
    name="multiply_two_numbers",
    description="Multiply two numbers together",
    parameters=parameters,
)


divide_two_numbers_func = FunctionDeclaration(
    name="divide_two_numbers",
    description="Divide first_number by second_number",
    parameters=parameters,
)

math_tool = Tool(
    function_declarations=[
        add_two_numbers_func,
        subtract_two_numbers_func,
        multiply_two_numbers_func,
        divide_two_numbers_func,
    ],
)

関数のレスポンスをシミュレートする代わりに、Gemini が関数呼び出しで応答したときに提供された引数を使用して呼び出す Python 関数を実際に記述します。

In [ ]:
# 数学ツールで使用される各関数宣言に対応する関数を定義
add_two_numbers = lambda a, b: a + b
subtract_two_numbers = lambda a, b: a - b
multiply_two_numbers = lambda a, b: a * b
divide_two_numbers = lambda a, b: a / b

次に、このツールのすべての関数呼び出しを処理する関数を定義します。

In [ ]:
def handle_math_fn_call(fn_name: str, fn_args: dict) -> Union[int, float]:
    """数学ツールの関数呼び出しを処理します。"""

    print(f"Function calling: {fn_name} with args: {fn_args}")
    a = fn_args["first_number"]
    b = fn_args["second_number"]

    if fn_name == "add_two_numbers":
        result = add_two_numbers(a, b)

    elif fn_name == "subtract_two_numbers":
        result = subtract_two_numbers(a, b)

    elif fn_name == "multiply_two_numbers":
        result = multiply_two_numbers(a, b)

    elif fn_name == "divide_two_numbers":
        result = divide_two_numbers(a, b)

    else:
        raise ValueError(f"Unknown function call: {fn_name}")

    return result

モデルとチャットエージェントをインスタンス化し、いくつかクエリをテストしてみましょう。

In [ ]:
chat = ChatAgent(tools=[math_tool], tool_handler_fn=handle_math_fn_call)

In [ ]:
response = chat.send_message("What is one plus one?")
response.text

In [ ]:
response = chat.send_message("Thanks! What is (5 * 5) / (4 + 1)?")
response.text

Gemini が質問に答えるために、論理的かつ順番に複数の関数を呼び出していることに注目してください。素晴らしいですね。

## データベース実行を伴う自然言語から SQL への変換（Natural Language to SQL）
Function Calling は、SQL の生成と実行を必要とし、そのレスポンスを利用してクエリに回答するシステムで役立ちます。まずは BigQuery でデータセットを作成し、いくつかの公開テーブルをクローンすることから始めます。

In [ ]:
# データセットを作成
!bq mk --location="US" iowa_liquor_sales

公開データにクエリを実行してテーブルを作成します。

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE iowa_liquor_sales.sales AS 
SELECT * FROM `bigquery-public-data.iowa_liquor_sales.sales`

列の定義を含めるようにテーブルのスキーマを更新します。

In [ ]:
SCHEMA_FILE = "liquor_sales_schema.json"
!bq show --schema --format=prettyjson bigquery-public-data:iowa_liquor_sales.sales > {SCHEMA_FILE}
!bq update {PROJECT}:iowa_liquor_sales.sales {SCHEMA_FILE}

関数宣言を作成し、新しい `Tool` をインスタンス化します。関数宣言は次のとおりです。
1) 利用可能なテーブルを一覧表示する `list_available_tables`
2) 特定のテーブルに関する情報とスキーマを取得する `get_table_info`
3) ユーザーの質問に回答するために BigQuery から情報を取得する `sql_query`

In [ ]:
# テーブルが 1 つしかないため、トリガーされた場合はこの関数からのレスポンスをハードコードします
list_available_tables_func = FunctionDeclaration(
    name="list_available_tables",
    description="Get and list all available BigQuery tables with fully qualified IDs.",
    parameters=Schema(type="OBJECT", properties={}),
)

get_table_info_func = FunctionDeclaration(
    name="get_table_info",
    description="Get information about a BigQuery table and it's schema so you can better answer user questions.",
    parameters=Schema(
        type="OBJECT",
        properties={
            "table_id": Schema(
                type="STRING",
                description="Fully qualified ID of BigQuery table",
            )
        },
    ),
)


sql_query_func = FunctionDeclaration(
    name="sql_query",
    description="Get information from data in BigQuery using SQL queries",
    parameters=Schema(
        type="OBJECT",
        properties={
            "query": Schema(
                type="STRING",
                description=f"SQL query on a single line (no \\n characters) that will help give answers to users questions when run on BigQuery. Only query tables in project: {PROJECT}",
            )
        },
    ),
)

query_tool = Tool(
    function_declarations=[
        list_available_tables_func,
        get_table_info_func,
        sql_query_func,
    ],
)

次に、モデルがこれらの関数呼び出しのいずれかを返したときに実行される Python 関数を作成する必要があります。関数は次のように実装します。
* `list_available_tables`: パラメータを受け取らず、上記で作成した BigQuery テーブルの名前 `f"{PROJECT}.iowa_liquor_sales.sales"` を返します。
* `get_table_info`: `table_id` パラメータを受け取り、BigQuery クライアントライブラリを使用してテーブル情報とスキーマを取得します。
* `sql_query`: `query_str` パラメータを受け取り、BigQuery クライアントライブラリを使用して SQL クエリを実行し、結果を返します。

In [ ]:
def list_available_tables():
    return [f"{PROJECT}.iowa_liquor_sales.sales"]


def get_table_info(table_id: str) -> dict:
    """テーブル情報を含む BigQuery API からの辞書を返します"""
    bq_client = bigquery.Client()
    return bq_client.get_table(table_id).to_api_repr()


def sql_query(query_str: str):
    bq_client = bigquery.Client()
    try:
        # クエリ文字列を少しクリーンアップ
        query_str = (
            query_str.replace("\\n", "").replace("\n", "").replace("\\", "")
        )
        # print(query_str)
        query_job = bq_client.query(query_str)
        result = query_job.result()
        result = str([dict(x) for x in result])
        return result
    except Exception as e:
        return f"Error from BigQuery Query API: {str(e)}"

モデルから返された関数呼び出しを処理し、必要なロジックを呼び出す Python 関数を作成します。

In [ ]:
def handle_query_fn_call(fn_name: str, fn_args: dict):
    """クエリツールの関数呼び出しを処理します。"""

    print(f"Function calling: {fn_name} with args: {str(fn_args)}\n")
    if fn_name == "list_available_tables":
        result = list_available_tables()

    elif fn_name == "get_table_info":
        result = get_table_info(fn_args["table_id"])

    elif fn_name == "sql_query":
        result = sql_query(fn_args["query"])

    else:
        raise ValueError(f"Unknown function call: {fn_name}")

    return result

モデルとチャットエージェントをインスタンス化し、いくつかのクエリをテストしてみましょう。

In [ ]:
chat = ChatAgent(tools=[query_tool], tool_handler_fn=handle_query_fn_call)

In [ ]:
# モデルの動作や出力のスタイル/形式をガイドするために、最初のチャットの前に初期化プロンプトを挿入します

init_prompt = """
    Please give a concise and easy to understand answer to any questions. 
    Only use information that you learn by querying the BigQuery table. 
    Do not make up information. Be sure to look at which tables are available 
    and get the info of any relevant tables before trying to write a query. 
    
    Question:
"""

prompt = "Which store has sold the most bottles of all time?"
response = chat.send_message(init_prompt + prompt)
print(response.text)

In [ ]:
response = chat.send_message(
    "Interesting! What is the most popular bottle of all time?"
)
print(response.text)

In [ ]:
response = chat.send_message(
    "What are the five most popular bottles in polk county?"
)
print(response.text)

In [ ]:
response = chat.send_message(
    "What vendors have made the most revenue selling liquor?"
)
print(response.text)

生成された SQL を自由に実行して、レスポンスを検証・確認してみてください。簡単な方法は、先頭に `%%bigquery` を記述したコードセルで実行することです。例:

```
%%bigquery
SELECT ... 
FROM ... 
```

## エンティティ抽出のための Function Calling
これまでの例では、別の関数、API、またはクライアントライブラリにパラメータを渡すためにエンティティ抽出を使用しました。しかし、エンティティ抽出のステップのみを実行し、他のものを実際に呼び出さずにそこで終了させたい場合もあります。この機能は、非構造化テキストデータを構造化フィールドに変換するための便利な方法と考えることができます。

たとえば、未加工のログをエラーメッセージに関する詳細を含む構造化データに変換するログ抽出ツールを簡単に構築できます。

まず、関数宣言を指定することから始めます。

In [ ]:
extract_log_data_func = FunctionDeclaration(
    name="extract_log_data",
    description="Extracts specific details from errors in log data",
    parameters=Schema(
        type="OBJECT",
        properties={
            "errors": Schema(
                type="ARRAY",
                description="Errors",
                items=Schema(
                    type="OBJECT",
                    description="Details of the error",
                    properties={
                        "error_message": Schema(
                            type="STRING",
                            description="Full error message",
                        ),
                        "error_code": Schema(
                            type="STRING",
                            description="Error code",
                        ),
                        "error_type": Schema(
                            type="STRING",
                            description="Error type",
                        ),
                    },
                ),
            ),
        },
    ),
)

error_extraction_tool = Tool(
    function_declarations=[extract_log_data_func],
)

In [ ]:
prompt = """
[15:43:28] ERROR: Could not process image upload: Unsupported file format. (Error Code: 308)
[15:44:10] INFO: Search index updated successfully. 
[15:45:02] ERROR: Service dependency unavailable (payment gateway). Retrying... (Error Code: 5522) 
[15:45:33] ERROR: Application crashed due to out-of-memory exception. (Error Code: 9001) 
"""

response = client.models.generate_content(
    model=MODEL,
    config=GenerateContentConfig(tools=[error_extraction_tool]),
    contents=prompt,
)
function_calls = response.function_calls

In [ ]:
for func_call in function_calls:
    for err in dict(func_call.args).get("errors"):
        print(dict(err))

Function Calling は非常に汎用性の高いツールです。

## （オプション）LangChain による Function Calling
LangChain を使用して Function Calling を実装する方法を見てみましょう。

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

chat = ChatGoogleGenerativeAI(model="gemini-3.5-flash", vertexai=True)

LangChain で Function Calling を実装するには、LangChain の `tools` を利用します。これらのツールはスキーマによって定義され、通常の関数（型ヒントと docstring を含む）、[Pydantic モデル](https://python.langchain.com/docs/how_to/tool_calling/#pydantic-class)、[TypedDict クラス](https://python.langchain.com/docs/how_to/tool_calling/#typeddict-class)、専用の [LangChain Tool オブジェクト](https://python.langchain.com/api_reference/core/tools/langchain_core.tools.base.BaseTool.html#basetool) インスタンスなど、さまざまな Python 構成要素から作成できます。次の例は、Python 関数を使用してツールを定義する方法を示しており、各関数は `@tool` デコレータで修飾されています。

In [ ]:
@tool
def add(a: int, b: int) -> int:
    """2 つの整数を加算します。

    Args:
        a: 最初の整数
        b: 2 番目の整数
    """
    return a + b


@tool
def subtract(a: int, b: int) -> int:
    """first_number から second_number を減算します。

    Args:
        a: 最初の整数
        b: 2 番目の整数
    """
    return a - b


@tool
def multiply(a: int, b: int) -> int:
    """2 つの整数を乗算します。

    Args:
        a: 最初の整数
        b: 2 番目の整数
    """
    return a * b


@tool
def divide(a: int, b: int) -> int:
    """first_number を second_number で除算します。

    Args:
        a: 最初の整数
        b: 2 番目の整数
    """
    return a / b

Python 関数では、`@tool` デコレータを追加することでさらにカスタマイズできます。詳細については [ドキュメント](https://python.langchain.com/docs/how_to/custom_tools/#tool-decorator) を参照してください。

それでは、`.bind_tools()` メソッドを使用してツールをバインドしましょう。

In [ ]:
tools = [add, subtract, multiply, divide]

chat_with_tools = chat.bind_tools(tools)

これでモデルを呼び出し、モデルからのレスポンスを確認できます。<br>
モデルによって関数呼び出しが要求されると、レスポンスに `.tool_calls` が含まれます。

ここではチャット履歴を `messages` リストで管理していますが、複雑なエージェントシステムを構築する場合は LangGraph の使用を検討してください。

In [ ]:
query = "What is (5 * 5) / (4 + 1)?"
messages = [HumanMessage(query)]

response = chat_with_tools.invoke(messages)
messages.append(response)
response.tool_calls

各関数を呼び出して `messages` に追加しましょう。ツールからのレスポンスは [`ToolMessage`](https://api.python.langchain.com/en/latest/messages/langchain_core.messages.tool.ToolMessage.html) オブジェクトにラップされることに注意してください。

In [ ]:
for tool_call in response.tool_calls:
    selected_tool = {
        "add": add,
        "subtract": subtract,
        "multiply": multiply,
        "divide": divide,
    }[tool_call["name"]]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

関数のレスポンスを LLM モデルに提供し、再度呼び出します。

In [ ]:
response = chat_with_tools.invoke(messages)
messages.append(response)
response.tool_calls

モデルは別の関数呼び出しで応答しました。同じプロセスをもう一度繰り返しましょう。

In [ ]:
for tool_call in response.tool_calls:
    selected_tool = {
        "add": add,
        "subtract": subtract,
        "multiply": multiply,
        "divide": divide,
    }[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

chat_with_tools.invoke(messages)

これで最終的なレスポンスを正常に取得できました。

Copyright 2024 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.